# M3 – Step 5: Oracle request 2 by scoring (exploitation / exploration / random / repeat)

Budget: 30 points in at most 3 requests. Request 1 used 15. Request 2 uses **9** and keeps **6** for request 3.

Split (my choice): **3 exploitation + 3 exploration + 2 random + 1 repeat**.

| Category | What it is for | How it is chosen |
|---|---|---|
| Exploitation | FM where M1 actually needs it (hover thrust of designs A–E, loaded and empty) | highest *exploitation score* |
| Exploration | where the surrogate knows least | highest *exploration score* |
| Random | unbiased check points, not chosen by the model | uniform over the box, fixed seed |
| Repeat | oracle noise (no repeats in request 1) | repeat of pt 3 (worst LOO, steep high-DL drop) |

**Scores** (each in [0, 1]; code and weights in `src/m3_surrogate/select_points.py`):
- exploration = 0.7 · σ/σmax + 0.3 · |RBF cubic − GP| / max
- exploitation = closeness to M1 hover points × σ/σmax, closeness = max over hover points of
  weight · exp(−d²/(2·0.08²)), d in scaled (D, T); weight 1 for baseline C, 0.6 for A, B, D, E.

**Greedy rule:** pick the highest score among candidates at least 0.10 (scaled) from every existing and chosen point;
after each pick recompute σ with that point added (GP hyperparameters fixed; σ does not need the FM value).

Write your **prediction before running** and **observation after** for each section.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from m3_surrogate import load_oracle, write_request_csv
from m3_surrogate import select_points as sp

STUDENT_ID = 'ROLL_NUMBER'      # <-- your roll number before sending
ORACLE_DIR = ROOT / 'data' / 'oracle'
FIG_DIR = ROOT / 'figures'; FIG_DIR.mkdir(exist_ok=True)
RES_DIR = ROOT / 'results'; RES_DIR.mkdir(exist_ok=True)
pd.set_option('display.width', 200)

df = load_oracle(ORACLE_DIR)
assert set(df.batch) == {'result_1'}, "planning request 2: only result_1 should be loaded"
designs = pd.read_csv(RES_DIR / 'm1_designs.csv')          # written by M1 (run_all.py --only m1)
print(f"{len(df)} oracle points; M1 designs: {list(designs.design)}")
print(f"weights: W_SIGMA={sp.W_SIGMA}, W_DISAGREE={sp.W_DISAGREE}, ELL={sp.ELL}, "
      f"C weight={sp.DESIGN_WEIGHT['C']}, others={sp.OTHER_WEIGHT}, D_MIN={sp.D_MIN}, seed={sp.SEED}")

## 1. The M1 hover points (where M1 reads FM)

Loaded T = m·g/4, empty T = (m − 2.5)·g/4 for each design. `dist_existing` = scaled distance to the nearest
request-1 point: below 0.10 means the point is already (almost) measured.

Prediction (which hover points are already covered by request 1?): _..._

In [ ]:
import numpy as np
from m3_surrogate import make_features, disk_loading
X1 = make_features(df.D_m, df.T_N)
hover = sp.m1_hover_points(designs)
Xh = make_features(hover.D_m, hover.T_N)
d = np.linalg.norm(Xh[:, None, :] - X1[None, :, :], axis=2)
hover['DL'] = disk_loading(hover.D_m, hover.T_N)
hover['nearest_pt'] = df.index[d.argmin(axis=1)]
hover['dist_existing'] = d.min(axis=1)
hover.round(3)

Observation: _..._

## 2. Score the grid and choose the 9 points

Prediction (where will the exploration picks go: the DL gap, or the box corners?): _..._

In [ ]:
plan, grid = sp.choose_request(df, designs)
plan.drop(columns='why').round(3)

Columns: scores and σ are at the moment of the pick (after earlier picks); `dist_existing` = to request 1,
`dist_any` = to request 1 + earlier picks; `FM_GP ± sd_GP` and `FM_RBF` = what the models predict now
(compare with `result_2.csv` when it arrives).

Observation: _..._

In [ ]:
fig = sp.plot_scores(df, plan, grid, path=FIG_DIR / 'm3_request2_scores.png')

Observation (did the distance rule remove the near-duplicates A-empty ≈ pt 15 and E-loaded ≈ pt 2?
are any picks in the DL gap?): _..._

## 3. Write the request file and the plan table

In [ ]:
req = write_request_csv(plan, ORACLE_DIR / 'request_2.csv', student_id=STUDENT_ID)
plan.round(4).to_csv(RES_DIR / 'm3_request2_plan.csv')
print(req.to_string(index=False))
print(f"\nEmail subject:  ED6002 ORACLE {STUDENT_ID} 2")

When the result comes back, save it as `data/oracle/result_2.csv`; `load_oracle` picks it up automatically.

## 4. Defence (in your own words, for the decision log)
1. Why this split (3/3/2/1) and why keep 6: _..._
2. Exploitation picks and the M1 designs they serve: _..._
3. Exploration picks (why corners/edges, not the DL gap): _..._
4. Random picks (why they make the next LOO / error check fairer): _..._
5. Repeat (what difference would mean "noisy"): _..._